In [21]:
from sklearn.metrics import classification_report
from sentence_transformers import CrossEncoder
from datasets import load_dataset
import numpy as np
# 0.4B

In [ ]:
model = CrossEncoder('cross-encoder/nli-deberta-v3-large')

In [2]:
data = [
    ("(John, PerformsAction, born), (born, AtLocation, spain)","(John, birth-place, Spain)"), #entailement
    ("(John, PerformsAction, eating), (pizza, ReceivesAction, eating)","(John, eats, ham pizza)"), # neutral
    ("(John, PerformsAction, eating), (pizza, ReceivesAction, eating), (pizza, Contains, ham)","(John, eats, ham pizza)") # entailement

    ]
scores = model.predict(data)

In [3]:
label_mapping = ['contradiction', 'entailment', 'neutral']
labels = [label_mapping[score_max] for score_max in scores.argmax(axis=1)]

In [4]:
labels

['entailment', 'neutral', 'contradiction']

In [47]:
anli_test = load_dataset("facebook/anli", split="test_r1")
y = anli_test["label"]
remap = {2: 0, 0: 1, 1: 2}
y = [remap[label] for label in anli_test["label"]]


In [48]:
anli_test.shape

(1000, 5)

In [49]:
pairs = [(line["premise"], line["hypothesis"]) for line in anli_test]

In [50]:
anli_test[1]

{'uid': 'c577b92c-78fb-4e1d-ae1d-34133609c142',
 'premise': 'Old Trafford is a football stadium in Old Trafford, Greater Manchester, England, and the home of Manchester United. With a capacity of 75,643, it is the largest club football stadium in the United Kingdom, the second-largest football stadium, and the eleventh-largest in Europe. It is about 0.5 mi from Old Trafford Cricket Ground and the adjacent tram stop.',
 'hypothesis': 'There are only 10 larger football stadiums in Europe.',
 'label': 0,
 'reason': 'The text says that it is the 11th largest football stadium in Europe.'}

In [51]:
y_prob = model.predict(pairs)
y_pred = np.argmax(y_prob, axis=1)

In [52]:
print(classification_report(y, y_pred, target_names=label_mapping))

               precision    recall  f1-score   support

contradiction       0.56      0.51      0.54       333
   entailment       0.56      0.51      0.54       334
      neutral       0.48      0.56      0.52       333

     accuracy                           0.53      1000
    macro avg       0.53      0.53      0.53      1000
 weighted avg       0.53      0.53      0.53      1000

